# Lab Apache Iceberg

## Objetivo

Usar Apache Iceberg con un dataset NYC TLC:
- catálogo
- creación de tablas
- inserción
- snapshots
- evolución de esquema
- consultas por partición lógica


In [1]:
# dependencias
#!pip install pyspark==4.0.2

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

warehouse_path = "/content/iceberg_warehouse"

spark = (
    SparkSession.builder
    .appName("EAFIT-Colab-PySpark-4.0.2-Iceberg-NYC-TLC")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", warehouse_path)
    .config("spark.sql.defaultCatalog", "local")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Warehouse:", warehouse_path)

Spark version: 4.0.2
Warehouse: /content/iceberg_warehouse


# tablas sencillas de prueba

In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db")
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.db.ejemplo_iceberg (
        id BIGINT,
        nombre STRING
    ) USING iceberg
""")

spark.sql("INSERT INTO local.db.ejemplo_iceberg VALUES (1, 'Ana'), (2, 'Luis')")
spark.sql("SELECT * FROM local.db.ejemplo_iceberg").show()

+---+------+
| id|nombre|
+---+------+
|  1|   Ana|
|  2|  Luis|
+---+------+



# descargar datasets

In [4]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
DATASET = "yellow"   # opciones: yellow, green, fhv, hvfhv
YEAR = 2025
MONTHS = [1, 2, 3]   # ampliar a 6 o 12 meses para más volumen

def build_urls(dataset=DATASET, year=YEAR, months=MONTHS):
    return [
        f"{BASE_URL}/{dataset}_tripdata_{year}-{m:02d}.parquet"
        for m in months
    ]

urls = build_urls()
urls


['https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-03.parquet']

In [5]:

import os
import urllib.request
from pathlib import Path

raw_dir = Path("datalake/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

def download_if_missing(url, output_dir=raw_dir):
    filename = url.split("/")[-1]
    target = output_dir / filename
    if not target.exists():
        print(f"Descargando {filename} ...")
        urllib.request.urlretrieve(url, target)
    else:
        print(f"Ya existe: {filename}")
    return str(target)

local_files = [download_if_missing(u) for u in urls]
local_files[:3]


Descargando yellow_tripdata_2025-01.parquet ...
Descargando yellow_tripdata_2025-02.parquet ...
Descargando yellow_tripdata_2025-03.parquet ...


['datalake/raw/yellow_tripdata_2025-01.parquet',
 'datalake/raw/yellow_tripdata_2025-02.parquet',
 'datalake/raw/yellow_tripdata_2025-03.parquet']

# preparar los datasets y crear la bd local.taxis

In [6]:

df = (
    spark.read.parquet(*local_files)
    .withColumn("year", F.year("tpep_pickup_datetime"))
    .withColumn("month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
)

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.taxis")


DataFrame[]

In [7]:

df.createOrReplaceTempView("staging_taxi")

spark.sql("""
CREATE OR REPLACE TABLE local.taxis.yellow_trips
USING iceberg
PARTITIONED BY (year, month)
AS
SELECT *
FROM staging_taxi
""")


DataFrame[]

In [8]:

spark.sql("SELECT year, month, count(*) AS n FROM local.taxis.yellow_trips GROUP BY year, month ORDER BY year, month").show()


+----+-----+-------+
|year|month|      n|
+----+-----+-------+
|2007|   12|      1|
|2009|    1|      1|
|2024|   12|     21|
|2025|    1|3253872|
|2025|    2|3310812|
|2025|    3|3848643|
|2025|    4|      2|
+----+-----+-------+



## Evolución de esquema

In [9]:
spark.sql("ALTER TABLE local.taxis.yellow_trips ADD COLUMN ingestion_source string")
spark.sql("UPDATE local.taxis.yellow_trips SET ingestion_source = 'nyc_tlc_official'")
spark.sql("SELECT ingestion_source, count(*) FROM local.taxis.yellow_trips GROUP BY ingestion_source").show()


+----------------+--------+
|ingestion_source|count(1)|
+----------------+--------+
|nyc_tlc_official|10413352|
+----------------+--------+



## Snapshots y metadatos

In [10]:
spark.sql("SELECT * FROM local.taxis.yellow_trips.snapshots").show(truncate=False)


+-----------------------+-------------------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                              |summary                        

In [11]:
spark.sql("SELECT * FROM local.taxis.yellow_trips.files").show(truncate=False)


+-------+-----------------------------------------------------------------------------------------------------------------------------------+-----------+-------+----------+------------+------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------

## Inserción incremental de otro mes

In [12]:
extra_urls = [f"{BASE_URL}/{DATASET}_tripdata_{YEAR}-04.parquet"]
extra_files = [download_if_missing(u) for u in extra_urls]
extra_df = (
    spark.read.parquet(*extra_files)
    .withColumn("year", F.year("tpep_pickup_datetime"))
    .withColumn("month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
)
extra_df.createOrReplaceTempView("extra_taxi")

spark.sql("""
INSERT INTO local.taxis.yellow_trips
SELECT *, 'nyc_tlc_official' AS ingestion_source
FROM extra_taxi
""")


Descargando yellow_tripdata_2025-04.parquet ...


DataFrame[]

In [13]:
spark.sql("SELECT year, month, count(*) AS n FROM local.taxis.yellow_trips GROUP BY year, month ORDER BY year, month").show()
spark.sql("SELECT * FROM local.taxis.yellow_trips.snapshots").show(truncate=False)


+----+-----+-------+
|year|month|      n|
+----+-----+-------+
|2007|   12|      1|
|2009|    1|      1|
|2024|   12|     21|
|2025|    1|3253872|
|2025|    2|3310812|
|2025|    3|3848647|
|2025|    4|3706102|
|2025|    5|      3|
+----+-----+-------+

+-----------------------+-------------------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------